In [10]:
# UAM vs TNC Market Share Estimation using MNL

import numpy as np
import pandas as pd
from geopy.distance import geodesic

# ----------------------------
# Step 1: Setup Coordinates from Example.ipynb
# ----------------------------
coordinates = {
    "LAX": (33.94417072663332, -118.40248449470265),
    "DTLA": (34.043119342992945, -118.2672609267252),
    "LGB": (33.81600638045284, -118.15121229608207),
    "WDHL": (34.17106155814583, -118.6052759222859),
    "UVS": (34.13843197335217, -118.35511138411983),
    "ANH": (33.81221610988904, -117.91897753216998),
    "HWD": (34.091746181098755, -118.32725521959766),
    "PSD": (34.08807589928915, -118.55063573741867),
    "BVH": (34.06666969690935, -118.4114352069469),
}

vertiports = list(coordinates.keys())
vertiports.remove("LAX")  # LAX is the hub

# ----------------------------
# Step 2: Compute Distance Matrix (miles)
# ----------------------------
distance_miles = {
    vp: geodesic(coordinates["LAX"], coordinates[vp]).miles
    for vp in vertiports
}

# ----------------------------
# Step 3: Define Assumptions
# ----------------------------
uam_speed_mph = 150
uam_ingress = 10    # min
uam_wait = 2.5     # min
uam_egress = 8     # min
uam_cost_per_mile = 3  # USD per mile

tnc_speed_mph = 30
uber_cost_per_mile = 2.5  # USD per mile

# ----------------------------
# Step 4: Compute Travel Times and Costs
# ----------------------------
data = []
for vp, dist in distance_miles.items():
    tnc_time = dist / tnc_speed_mph * 60
    uam_flight = dist / uam_speed_mph * 60
    uam_time = uam_flight + uam_ingress + uam_wait + uam_egress

    uam_cost = dist * uam_cost_per_mile
    tnc_cost = dist * uber_cost_per_mile

    data.append({
        "destination": vp,
        "distance_mi": dist,
        "tnc_time_min": tnc_time,
        "tnc_cost_usd": tnc_cost,
        "uam_time_min": uam_time,
        "uam_cost_usd": uam_cost,
    })

results_df = pd.DataFrame(data)

# ----------------------------
# Step 5: MNL Mode Share Model
# ----------------------------
#beta coefficients for time and cost for non-business travelers based on Hotle study
beta_time = -0.05
beta_cost = -0.1

U_uam = beta_time * results_df["uam_time_min"] + beta_cost * results_df["uam_cost_usd"]
U_tnc = beta_time * results_df["tnc_time_min"] + beta_cost * results_df["tnc_cost_usd"]

exp_uam = np.exp(U_uam)
exp_tnc = np.exp(U_tnc)

results_df["P_uam"] = exp_uam / (exp_uam + exp_tnc)
results_df["P_tnc"] = exp_tnc / (exp_uam + exp_tnc)

# ----------------------------
# Step 6: Output Market Shares
# ----------------------------
results_df = results_df[[
    "destination", "distance_mi",
    "uam_time_min", "uam_cost_usd", "P_uam",
    "tnc_time_min", "tnc_cost_usd", "P_tnc"
]]

results_df.sort_values("P_uam", ascending=False, inplace=True)
results_df.reset_index(drop=True, inplace=True)

results_df


,destination,distance_mi,uam_time_min,uam_cost_usd,P_uam,tnc_time_min,tnc_cost_usd,P_tnc
0,ANH,29.245414,32.198166,87.736242,0.463158,58.490828,73.113535,0.536842
1,WDHL,19.490885,28.296354,58.472655,0.391676,38.981770,48.727212,0.608324
2,LGB,16.931477,27.272591,50.794431,0.373541,33.862954,42.328693,0.626459
3,UVS,13.662465,25.964986,40.987394,0.350891,27.324929,34.156161,0.649109
4,PSD,13.064427,25.725771,39.193282,0.346815,26.128855,32.661069,0.653185
5,HWD,11.049968,24.919987,33.149904,0.333255,22.099936,27.624920,0.666745
6,DTLA,10.333313,24.633325,30.999940,0.328495,20.666627,25.833283,0.671505
7,BVH,8.458742,23.883497,25.376225,0.316211,16.917483,21.146854,0.683789


In [16]:
# ----------------------------------------
# Step 7: Calculate UAM Demand Vector
# ----------------------------------------

# Total directional demand (same as n network.load_demand)
directional_demand = 4000

# Vector of UAM demand per destination based on P_uam
uam_demand_vector = directional_demand * results_df["P_uam"].values
results_df["uam_demand"] = np.round(uam_demand_vector, 2)

# ----------------------------------------
# Step 8: Normalize to get new vertiport_pmf
# ----------------------------------------

# Convert demand to PMF (probability mass function)
vertiport_pmf = uam_demand_vector / uam_demand_vector.sum()

# Display
results_df["pmf"] = np.round(vertiport_pmf, 4)
results_df = results_df[[
    "destination", "distance_mi",
    "uam_time_min", "uam_cost_usd", "P_uam", "uam_demand", "pmf",
    "tnc_time_min", "tnc_cost_usd", "P_tnc"
]]

results_df


,destination,distance_mi,uam_time_min,uam_cost_usd,P_uam,uam_demand,pmf,tnc_time_min,tnc_cost_usd,P_tnc
0,ANH,29.245414,32.198166,87.736242,0.463158,1852.63,0.1595,58.490828,73.113535,0.536842
1,WDHL,19.490885,28.296354,58.472655,0.391676,1566.70,0.1349,38.981770,48.727212,0.608324
2,LGB,16.931477,27.272591,50.794431,0.373541,1494.16,0.1286,33.862954,42.328693,0.626459
3,UVS,13.662465,25.964986,40.987394,0.350891,1403.56,0.1208,27.324929,34.156161,0.649109
4,PSD,13.064427,25.725771,39.193282,0.346815,1387.26,0.1194,26.128855,32.661069,0.653185
5,HWD,11.049968,24.919987,33.149904,0.333255,1333.02,0.1148,22.099936,27.624920,0.666745
6,DTLA,10.333313,24.633325,30.999940,0.328495,1313.98,0.1131,20.666627,25.833283,0.671505
7,BVH,8.458742,23.883497,25.376225,0.316211,1264.85,0.1089,16.917483,21.146854,0.683789


In [18]:
# Insert 0 for LAX (origin)
vertiport_pmf_full = np.insert(vertiport_pmf, 0, 0)  # Insert at position 0

# Save corrected PMF
np.save("vertiport_pmf_mnl.npy", vertiport_pmf_full)
